# End-to-End Demo
This notebook walks through each layer of the pipeline in sequence:
1. Raw JSON ingestion output
2. Processed Parquet data
3. Analytics outputs

In [27]:
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import glob

BASE_PATH = "/Users/snehamungre/projects/crypto_market_analysis"
TODAY_DATE = "2026-06-02"

spark = SparkSession.builder \
    .appName("CryptoDemo") \
    .config("spark.sql.warehouse.dir", f"{BASE_PATH}/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session started successfully")

Spark session started successfully


---
## Stage 1 — Raw Ingestion Output

In [2]:
raw_files = sorted(glob.glob(f"{BASE_PATH}/data/raw/*.json"))
latest_raw = raw_files[-1]

print(f"Most recent raw file: {os.path.basename(latest_raw)}")
print(f"Total raw snapshots accumulated: {len(raw_files)}\n")

with open(latest_raw, "r") as f:
    raw_data = json.load(f)

print(f"Number of coins in latest snapshot: {len(raw_data)}")
print("\nSample record (first coin):")
print(json.dumps(raw_data[0], indent=2))

Most recent raw file: crypto_market_data_raw_2026-06-02.json
Total raw snapshots accumulated: 7

Number of coins in latest snapshot: 100

Sample record (first coin):
{
  "id": "bitcoin",
  "symbol": "btc",
  "name": "Bitcoin",
  "image": "https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400",
  "current_price": 70506,
  "market_cap": 1413365565210,
  "market_cap_rank": 1,
  "fully_diluted_valuation": 1413365565210,
  "total_volume": 50009922040,
  "high_24h": 73343,
  "low_24h": 70120,
  "price_change_24h": -2799.818013942655,
  "price_change_percentage_24h": -3.81939,
  "market_cap_change_24h": -55207571774.16846,
  "market_cap_change_percentage_24h": -3.75927,
  "circulating_supply": 20037690.0,
  "total_supply": 20037690.0,
  "max_supply": 21000000.0,
  "ath": 126080,
  "ath_change_percentage": -44.07867,
  "ath_date": "2025-10-06T18:57:42.558Z",
  "atl": 67.81,
  "atl_change_percentage": 103876.68392,
  "atl_date": "2013-07-06T00:00:00.000Z",
  "roi": null,

---
## Stage 2 — Processed Parquet Data

In [3]:
processed_df = spark.read.parquet(f"{BASE_PATH}/data/processed")


print(f"Total records in processed layer: {processed_df.count()}")
print(
    f"Total records in processed layer for date {TODAY_DATE}: {processed_df[processed_df['updated_date'] == TODAY_DATE].count()}"
)
print(f"Number of partitions (dates): {processed_df.select('updated_date').distinct().count()}")
print("\nSchema:")
processed_df.printSchema()

Total records in processed layer: 403
Total records in processed layer for date 2026-06-02: 99


Number of partitions (dates): 11

Schema:
root
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_percentage: double (nullable = true)
 |-- atl_date: timestamp (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- current_price: double (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- name: string (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = tr

In [4]:
print("Dates available in processed layer:")
processed_df.select("updated_date").distinct().orderBy("updated_date").show(truncate=False)

print("Sample processed records:")
processed_df.select(
    "name", "current_price", "market_cap", "total_volume",
    "circulating_supply", "updated_date"
).orderBy("market_cap", ascending=False).show(15, truncate=False)

Dates available in processed layer:
+------------+
|updated_date|
+------------+
|2026-05-18  |
|2026-05-21  |
|2026-05-23  |
|2026-05-24  |
|2026-05-25  |
|2026-05-26  |
|2026-05-28  |
|2026-05-29  |
|2026-05-31  |
|2026-06-01  |
|2026-06-02  |
+------------+

Sample processed records:


+--------+-------------+-------------+---------------+--------------------+------------+
|name    |current_price|market_cap   |total_volume   |circulating_supply  |updated_date|
+--------+-------------+-------------+---------------+--------------------+------------+
|Bitcoin |77910.0      |1560349334869|2.9194154601E10|2.00322E7           |2026-05-21  |
|Bitcoin |77265.0      |1547282708175|2.4931599587E10|2.0034078E7         |2026-05-25  |
|Bitcoin |73306.0      |1469300843946|3.239469181E10 |2.003599E7          |2026-05-29  |
|Bitcoin |70506.0      |1413365565210|5.000992204E10 |2.003769E7          |2026-06-02  |
|Ethereum|2140.3       |258293665100 |1.1985654099E10|1.206856184990976E8 |2026-05-21  |
|Ethereum|2110.73      |254547653212 |1.1424936404E10|1.206855183533083E8 |2026-05-25  |
|Ethereum|2001.94      |241700370642 |1.3848108874E10|1.206853496009245E8 |2026-05-29  |
|Ethereum|1989.24      |240334060880 |1.6938630615E10|1.206851384286977E8 |2026-06-02  |
|Tether  |0.998993   

---
## Stage 4 — Analytics Outputs
The analytics job produces five output tables. We load and display each one below.

In [5]:
print("Average Market Cap Rankings:")
avgMarketCap = spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_market_cap").orderBy(
    "avg_market_cap_rank"
)

avgMarketCap.show(10, truncate=False)

Average Market Cap Rankings:
+------------+-----------------+-------------------+
|name        |avg_market_cap   |avg_market_cap_rank|
+------------+-----------------+-------------------+
|Bitcoin     |1.49757461305E12 |1                  |
|Ethereum    |2.487189374585E11|2                  |
|Tether      |1.890757690125E11|3                  |
|BNB         |8.87599313195E10 |4                  |
|XRP         |8.224263383375E10|5                  |
|USDC        |7.615612972825E10|6                  |
|Solana      |4.8321397802E10  |7                  |
|TRON        |3.3577417804E10  |8                  |
|Figure Heloc|1.87306852412E10 |9                  |
|Dogecoin    |1.5691272789E10  |10                 |
+------------+-----------------+-------------------+
only showing top 10 rows


In [6]:
print("Average Price Rankings:")
avgPriceRank = spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_price") \
    .orderBy("avg_price_rank") 
avgPriceRank.show(10, truncate=False)

Average Price Rankings:
+------------+-------------+--------------+
|name        |average_price|avg_price_rank|
+------------+-------------+--------------+
|Bitcoin     |74746.75     |1             |
|PAX Gold    |4524.233     |2             |
|Tether Gold |4515.918     |3             |
|Ethereum    |2060.553     |4             |
|BNB         |658.323      |5             |
|Zcash       |607.655      |6             |
|Monero      |376.795      |7             |
|Bitcoin Cash|331.183      |8             |
|Bittensor   |265.348      |9             |
|OUSG        |115.343      |10            |
+------------+-------------+--------------+
only showing top 10 rows


In [21]:
print("Volume to Market Cap Ratio Rankings:")
volumetoMarket = spark.read.parquet(f"{BASE_PATH}/data/analytics/vol_market_ratio") \
    .orderBy("vol_market_rank") 

volumetoMarket.show(10, truncate=False)

Volume to Market Cap Ratio Rankings:
+-------------------------------------+----------------+---------------+---------------+
|name                                 |vol_market_ratio|total_volume   |vol_market_rank|
+-------------------------------------+----------------+---------------+---------------+
|USD1                                 |0.46663         |2.176174676E9  |1              |
|Artificial Superintelligence Alliance|0.45314         |2.85592056E8   |2              |
|Worldcoin                            |0.43743         |6.86122707E8   |3              |
|Tether                               |0.41038         |7.7125762905E10|4              |
|Humanity                             |0.40408         |5.81601902E8   |5              |
|Dash                                 |0.39574         |2.50219036E8   |6              |
|Injective                            |0.35359         |2.46146748E8   |7              |
|NEAR Protocol                        |0.31641         |1.062448223E9  |8

In [8]:
print("Current Top Coins by Price (latest snapshot):")
currentTopCoins = (
    spark.read.parquet(f"{BASE_PATH}/data/analytics/curr_top_price")\
    .filter(f"updated_date == '{TODAY_DATE}'")\
    .orderBy("current_price_rank")
)

currentTopCoins.show(10, truncate=False)

Current Top Coins by Price (latest snapshot):
+------------------+-------------+-------------+---------------+---------------+------------------+------------+
|name              |current_price|market_cap   |market_cap_rank|total_volume   |current_price_rank|updated_date|
+------------------+-------------+-------------+---------------+---------------+------------------+------------+
|Bitcoin           |70506.0      |1413365565210|1              |5.000992204E10 |1                 |2026-06-02  |
|PAX Gold          |4507.98      |2092392931   |43             |1.55782833E8   |2                 |2026-06-02  |
|Tether Gold       |4494.96      |2754262594   |38             |2.2240362E8    |3                 |2026-06-02  |
|Ethereum          |1989.24      |240334060880 |2              |1.6938630615E10|4                 |2026-06-02  |
|BNB               |684.48       |92339834079  |4              |2.532623788E9  |5                 |2026-06-02  |
|Zcash             |556.74       |9305742533   |13

In [9]:
print("Top Performing Assets — Composite Ranking:")
topPerformingAssetsComp = spark.read.parquet(f"{BASE_PATH}/data/analytics/top_performing_assets") \
    .select(
        "top_performing_rank", "name", "avg_market_cap",
        "avg_market_cap_rank", "average_price", "avg_price_rank",
        "total_volume", "vol_market_ratio", "vol_market_rank",
        "top_performing_score"
    ) \
    .orderBy("top_performing_rank") \

topPerformingAssetsComp.show(10, truncate=False)

Top Performing Assets — Composite Ranking:
+-------------------+------------+-----------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|top_performing_rank|name        |avg_market_cap   |avg_market_cap_rank|average_price|avg_price_rank|total_volume   |vol_market_ratio|vol_market_rank|top_performing_score|
+-------------------+------------+-----------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|1                  |Ethereum    |2.487189374585E11|2                  |2060.553     |4             |1.6938630615E10|0.07048         |41             |11.0                |
|2                  |Bitcoin     |1.49757461305E12 |1                  |74746.75     |1             |5.000992204E10 |0.03538         |60             |13.0                |
|3                  |Zcash       |1.0132983297E10  |13                 |607.655      |6          

---
## Stage 5 — Visualization 

In [32]:
import plotly.express as px

avg_market_df = pd.read_parquet(f"{BASE_PATH}/data/analytics/avg_market_cap")

# Sort and slice the top 10
top_10_market = avg_market_df.sort_values(by="avg_market_cap", ascending=False).head(10)

# Plotly handles the slicing, percentages, and labels completely out of the box!
fig = px.pie(
    top_10_market,
    values="avg_market_cap",
    names="name",
    title="Average Market Capitalization Share of Top 10 Cryptocurrencies",
)

# Force labels to pull outward cleanly with dynamic connectors if they get crowded
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()

In [49]:
# 2. Read the entire historical processed directory
processed_df = pd.read_parquet(f"{BASE_PATH}/data/processed")

# 3. Filter down to the specific assets you want to analyze
target_coins = [
    "Bitcoin",
    "Ethereum",
    "Zcash",
    "BNB",
    "Solana",
    "Hyperliquid",
    "Bitcoin Cash",
    "Monero",
    "PAX Gold",
    "Tether Gold",
]
timeline_df = processed_df[processed_df["name"].isin(target_coins)].copy()

# 4. CRITICAL: Sort chronologically by date first so that 'first' represents Day 1
timeline_df = timeline_df.sort_values(by="updated_date")

# 5. THE MAGIC LINE: Group by each coin name, find its Day 1 price, and compute the normalized multiplier
timeline_df["price_index"] = timeline_df["current_price"] / timeline_df.groupby("name")[
    "current_price"
].transform("first")

# 6. Convert it to an explicit percentage ROI value for cleaner chart text labels
timeline_df["percentage_growth"] = (timeline_df["price_index"] - 1) * 100

# 7. Create the interactive normalized line chart
fig = px.line(
    timeline_df,
    x="updated_date",
    y="percentage_growth",  
    color="name",  
    markers=True,
    title="Cryptocurrency Price Comparison (% Change Since Day 1 Ingestion)",
    labels={
        "updated_date": "Timeline Date",
        "percentage_growth": "Cryptocurrency Price Comparison (%)",
        "name": "Asset Name",
    },
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="#333333",
    annotation_text="Initial Price (on Ingestion)",
)

# 9. Finalize presentation layout styling
fig.update_layout(
    template="plotly_white",
    yaxis_tickformat=".%"  # formats y axis text cleanly
)

fig.show()

In [41]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
